# DeepFake Sentinel — Phase 2 Training
## CNN-LSTM Temporal Detector (Video Face-Swap Detection)

This notebook fine-tunes the **CNN-LSTM** model from `backend/models/cnn_lstm.py` on short video clips, learning to detect **temporal inconsistencies** characteristic of face-swap deepfakes (flickering, jitter, unnatural transitions).

**Output:** `cnn_lstm.pt` — drop into `backend/weights/`. Combined with Phase 1's `efficientnet_b4.pt` + `vit_base.pt`, your `DetectionEngine` will run the full 3-model ensemble in real model mode (and `cnn_lstm_loaded` will become `true` for video uploads).

**Runtime:** `Runtime > Change runtime type > GPU` (T4).

---

## Data source

CNN-LSTM needs **labeled video clips** (real vs face-swapped). This notebook automatically downloads a small public dataset from Hugging Face: **`UniDataPro/deepfake-videos-dataset`** — 5 real + 5 deepfake clips, **no login, no account, no gating required** (publicly accessible files).

This is a small dataset (10 clips), so this notebook is configured to demonstrate the **full training pipeline working end-to-end** (data -> face extraction -> CNN-LSTM training -> export) rather than to achieve high accuracy. If you later get access to a larger labeled video dataset (FaceForensics++, Celeb-DF, etc.), you can swap the download cell below for that source — everything downstream (Sections 3-7) stays the same.

If you have your own real/fake `.mp4` clips, you can alternatively/additionally upload them — see the optional 'Upload your own clips' cell after the download cell.

## 1. Setup

In [ ]:
!pip install -q timm opencv-python-headless

In [ ]:
# Import your project's model + preprocessing code
from google.colab import files
import zipfile, os, sys

print("Upload your 'deepfake-detection-platform.zip'")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/project')

backend_dir = None
for root, dirs, _ in os.walk('/content/project'):
    if 'models' in dirs and os.path.basename(root) == 'backend':
        backend_dir = root
        break
assert backend_dir is not None, "Could not find backend/models in the zip"
sys.path.insert(0, os.path.dirname(backend_dir))
print("Found backend at:", backend_dir)

In [ ]:
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

try:
    from backend.models import create_cnn_lstm
    from backend.preprocess import VideoPreprocessor
    print("Imported via package successfully.")
except Exception as e:
    print("Package import failed (", e, ") — falling back to direct file import.")
    lstm_mod = load_module("cnn_lstm", os.path.join(backend_dir, "models", "cnn_lstm.py"))
    pre_mod = load_module("preprocess", os.path.join(backend_dir, "preprocess.py"))
    create_cnn_lstm = lstm_mod.create_cnn_lstm
    VideoPreprocessor = pre_mod.VideoPreprocessor
    print("Imported via direct file load successfully.")

## 2. Download real + fake video clips (DeepSpeak v2 public samples)

Downloads directly from Hugging Face — no authentication needed.

In [ ]:
import os
import urllib.request

os.makedirs("/content/data/real", exist_ok=True)
os.makedirs("/content/data/fake", exist_ok=True)

BASE_URL = "https://huggingface.co/datasets/UniDataPro/deepfake-videos-dataset/resolve/main/"

# 'video/' folder = real clips, 'deepfake/' folder = fake clips
REAL_FILES = [
    "video/1.mp4",
    "video/2.mp4",
    "video/3.mp4",
    "video/4.mp4",
    "video/5.MOV",
]

FAKE_FILES = [
    "deepfake/1.mp4",
    "deepfake/2.mp4",
    "deepfake/3.mp4",
    "deepfake/4.mp4",
    "deepfake/5.mov",
]

def download(rel_path, dest_dir):
    filename = os.path.basename(rel_path)
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest):
        return True
    url = BASE_URL + rel_path + "?download=true"
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print(f"  [skip] {rel_path}: {e}")
        return False

print("Downloading real clips...")
real_ok = sum(download(f, "/content/data/real") for f in REAL_FILES)
print("Downloading fake (deepfake) clips...")
fake_ok = sum(download(f, "/content/data/fake") for f in FAKE_FILES)

print(f"\nDownloaded: {real_ok} real, {fake_ok} fake clips.")
print("Real clips:", os.listdir("/content/data/real"))
print("Fake clips:", os.listdir("/content/data/fake"))

assert real_ok > 0 and fake_ok > 0, (
    "Download failed for all files. Check your internet connection or "
    "try the 'Upload your own clips' cell below instead."
)

### Optional: add your own clips

If you have additional real or fake `.mp4` clips, run the cell below to upload more (they'll be added alongside the downloaded DeepSpeak clips). **Skip this cell if the download above already worked and you're happy with 14 clips.**

In [ ]:
import shutil
from google.colab import files

print("Upload additional REAL video clips (or click Cancel to skip).")
real_uploads = files.upload()
for fname in real_uploads:
    shutil.move(fname, os.path.join("/content/data/real", fname))

print("\nUpload additional FAKE / deepfake video clips (or click Cancel to skip).")
fake_uploads = files.upload()
for fname in fake_uploads:
    shutil.move(fname, os.path.join("/content/data/fake", fname))

print("\nReal clips:", os.listdir("/content/data/real"))
print("Fake clips:", os.listdir("/content/data/fake"))

## 3. Build frame-sequence dataset

For each video, extract `SEQ_LENGTH` evenly-spaced frames, detect+crop the primary face in each frame (reusing your project's `VideoPreprocessor`), and stack into a `(T, 3, 224, 224)` tensor. Clips are cached to disk as `.pt` files so re-runs are fast.

In [ ]:
import torch
from torchvision import transforms
from pathlib import Path

SEQ_LENGTH = 16   # must match cnn_lstm.py's seq_length default for consistency
IMG_SIZE = 224

preprocessor = VideoPreprocessor(target_size=IMG_SIZE, num_frames=SEQ_LENGTH)

normalize_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

CACHE_DIR = Path("/content/clip_cache")
CACHE_DIR.mkdir(exist_ok=True)


def build_clip_tensor(video_path: str, cache_name: str) -> torch.Tensor | None:
    """Extract SEQ_LENGTH face-crop frames from a video as a (T, 3, 224, 224) tensor.

    Returns None if no faces could be detected at all (clip is skipped).
    Caches the result to disk so re-running the notebook is fast.
    """
    cache_path = CACHE_DIR / f"{cache_name}.pt"
    if cache_path.is_file():
        return torch.load(cache_path)

    try:
        result = preprocessor.preprocess_pipeline(video_path)
    except Exception as e:
        print(f"  [skip] {video_path}: {e}")
        return None

    face_crops = result["face_crops"]
    frames_tensors = []
    for crops in face_crops:
        if crops:
            frames_tensors.append(normalize_tf(crops[0]))

    if len(frames_tensors) < 2:
        print(f"  [skip] {video_path}: not enough usable frames ({len(frames_tensors)})")
        return None

    # Pad or trim to exactly SEQ_LENGTH frames
    if len(frames_tensors) < SEQ_LENGTH:
        # repeat last frame to pad
        while len(frames_tensors) < SEQ_LENGTH:
            frames_tensors.append(frames_tensors[-1])
    else:
        frames_tensors = frames_tensors[:SEQ_LENGTH]

    clip = torch.stack(frames_tensors)  # (SEQ_LENGTH, 3, 224, 224)
    torch.save(clip, cache_path)
    return clip

In [ ]:
# Build the dataset: list of (clip_tensor, label) — label 1 = fake, 0 = real
samples = []

for label, cls in [(0, "real"), (1, "fake")]:
    cls_dir = Path("/content/data") / cls
    video_files = sorted(
        p for p in cls_dir.iterdir()
        if p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}
    )
    print(f"{cls}: {len(video_files)} videos found")

    for vp in video_files:
        clip = build_clip_tensor(str(vp), cache_name=f"{cls}_{vp.stem}")
        if clip is not None:
            samples.append((clip, label))

print(f"\nTotal usable clips: {len(samples)}")
n_real = sum(1 for _, l in samples if l == 0)
n_fake = sum(1 for _, l in samples if l == 1)
print(f"  real: {n_real}  |  fake: {n_fake}")

assert len(samples) >= 4, (
    "Need at least a few usable clips per class. "
    "Upload more videos, or check that faces are detectable in your clips."
)

## 4. Train / validation split + DataLoader

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

random.seed(42)
random.shuffle(samples)

split_idx = max(1, int(len(samples) * 0.8))
train_samples = samples[:split_idx]
valid_samples = samples[split_idx:] or samples[-1:]  # ensure non-empty

print(f"Train: {len(train_samples)} clips | Valid: {len(valid_samples)} clips")


class ClipDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        clip, label = self.items[idx]
        return clip, label


BATCH_SIZE = 2  # small clips are memory-heavy: (B, 16, 3, 224, 224)

train_loader = DataLoader(ClipDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(ClipDataset(valid_samples), batch_size=BATCH_SIZE, shuffle=False)

## 5. Train CNN-LSTM

With small datasets, the backbone is frozen by default (`freeze_backbone=True`) so only the LSTM + attention + classifier head are trained — this avoids overfitting and trains much faster.

In [ ]:
import torch.nn as nn
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = create_cnn_lstm(
    pretrained=True,
    num_classes=2,
    seq_length=SEQ_LENGTH,
    freeze_backbone=True,  # set False if you have a larger dataset and want full fine-tuning
).to(device)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 15  # small datasets need more epochs to converge
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    start = time.time()

    for clips, labels in train_loader:
        clips = clips.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(clips)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * clips.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for clips, labels in valid_loader:
            clips = clips.to(device)
            labels = labels.to(device)
            logits = model(clips)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / max(val_total, 1)
    elapsed = time.time() - start

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} "
          f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | {elapsed:.1f}s")

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "cnn_lstm_best.pt")
        print(f"  -> saved checkpoint (val_acc={val_acc:.4f})")

## 6. Sanity check: run the trained model on a few clips

In [ ]:
import torch.nn.functional as F

model.load_state_dict(torch.load("cnn_lstm_best.pt", map_location=device))
model.eval()

with torch.no_grad():
    for clip, label in samples[:6]:
        logits = model(clip.unsqueeze(0).to(device))
        fake_prob = F.softmax(logits, dim=-1)[0, 1].item()
        true_label = "FAKE" if label == 1 else "REAL"
        pred_label = "FAKE" if fake_prob >= 0.5 else "REAL"
        marker = "OK" if true_label == pred_label else "MISS"
        print(f"[{marker}] true={true_label:4s} pred={pred_label:4s} fake_prob={fake_prob*100:.1f}%")

## 7. Download the trained weights

Download `cnn_lstm_best.pt`, rename to `cnn_lstm.pt`, and place it in `backend/weights/` alongside `efficientnet_b4.pt` and `vit_base.pt` from Phase 1.

In [ ]:
from google.colab import files

if os.path.exists("cnn_lstm_best.pt"):
    files.download("cnn_lstm_best.pt")

print("\nAfter downloading:")
print("  1. Rename 'cnn_lstm_best.pt' -> 'cnn_lstm.pt'")
print("  2. Place it in backend/weights/ (alongside efficientnet_b4.pt and vit_base.pt)")
print("  3. Restart your server (python run.py)")
print("  4. Check /api/health -> metadata.cnn_lstm_loaded should be true for video uploads")

---
## Notes on data quality & expectations

- **Small datasets (10-40 clips/class)**: the model will learn *something* and the pipeline will be fully functional end-to-end, but real-world accuracy will be limited. This is fine for a portfolio demo — be transparent in your README about training set size, as recruiters value the honesty and understanding of the full pipeline (data -> preprocessing -> training -> serving) more than a single accuracy number.
- **Improving accuracy**: more clips, more diverse face-swap methods/sources, unfreezing the backbone (`freeze_backbone=False`) with a larger dataset, and longer training all help.
- **`cnn_lstm.pt` is optional** — your `DetectionEngine` (updated in Phase 1) runs fine with just `efficientnet_b4.pt` + `vit_base.pt` and approximates the temporal score. Adding `cnn_lstm.pt` completes the full 3-model ensemble as originally designed.